In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from openai import OpenAI

openai_client = OpenAI()

In [4]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)    

In [8]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Possibly — but it depends on the course’s enrollment rules and whether registration is still open.\n\nIf you want, I can help you figure it out. Usually you’ll need to check:\n- whether the course is still accepting students\n- any prerequisites or placement requirements\n- whether there’s a waitlist\n- the add/drop deadline\n- whether you need instructor approval\n\nIf you share the course name, platform, school, or a link, I can help you check the exact rules or draft a message asking to join.'

In [5]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [6]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [9]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"joining the course discovered late can I join after start enrollment new student"}', call_id='call_PIFXQryPqPSUOGWBkmfG9m5r', name='search', type='function_call', id='fc_0022d4136a75e891006a315905cfd4819d8a421df222cf33d7', namespace=None, status='completed')]

In [ ]:
call = response.output[0]

In [12]:
call

ResponseFunctionToolCall(arguments='{"query":"joining the course discovered late can I join after start enrollment new student"}', call_id='call_PIFXQryPqPSUOGWBkmfG9m5r', name='search', type='function_call', id='fc_0022d4136a75e891006a315905cfd4819d8a421df222cf33d7', namespace=None, status='completed')

In [ ]:
call.name

'search'

In [20]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [25]:
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "04919992b3",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "How should I start the course and follow the weekly workflow?",
    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).

In [26]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [27]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"joining the course discovered late can I join after start enrollment new student"}', call_id='call_PIFXQryPqPSUOGWBkmfG9m5r', name='search', type='function_call', id='fc_0022d4136a75e891006a315905cfd4819d8a421df222cf33d7', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_PIFXQryPqPSUOGWBkmfG9m5r',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflo

In [28]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join and start learning anytime.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.'

In [30]:
print(response.output_text)

Yes — you can still join and start learning anytime.

If you want a certificate, make sure you submit your project while submissions are still open.
